## Assignment 3: $k$ Nearest Neighbor


**Q1.**
1. What is the difference between regression and classification?
2. What is a confusion table? What does it help us understand about a model's performance?
3. What does the SSE quantify about a particular model?
4. What are overfitting and underfitting? 
5. Why does splitting the data into training and testing sets, and choosing $k$ by evaluating accuracy or SSE on the test set, improve model performance?
6. With classification, we can report a class label as a prediction or a probability distribution over class labels. Please explain the strengths and weaknesses of each approach.

**Q2.** This is a case study on $k$ nearest neighbor classification, using the `land_mines.csv` data.

The data consists of a label, `mine_type`, taking integer values 1 to 5, and three properties of the mine, `voltage`, `height` and `soil`. We want to predict the kind of mine from data about it. Imagine working for the DOD or a humanitarian aid agency, trying to help people remove land mines more safely.

1. Load the data. Perform some EDA, summarizing the target label and the features.
2. Split the sample 50/50 into training and test/validation sets. (The smaller the data are, the more equal the split should be, in my experience: Otherwise, all of the members of one class end up in the training or test data, and the model falls apart.)
3. Build a $k$-NN classifier. Explain how you select $k$.
4. Print a confusion table for the optimal model, comparing predicted and actual class label on the test set. How accurate is it? Where is performance more or less accurate?
5. Notice that you can have a lot of accurate predictions for a given type of mine, but still make a lot of mistakes. Please explain how you'd advise someone to actually use this predictive model in practice, given the errors that it tends to make.

---

## *Phase 1: Solution without AI

Complete all questions in the following cells without generative AI. Then commit and push the
notebook before beginning Phase 2.

**Declaration:** I completed this version without generative AI.

**Student(s):** Annie Nguyen

**Date:** 10/2/2026

In [ ]:
# Your Phase 1 solution to the problem goes here.

# Question 1
1. The difference between regression and classification is the prediction it produces. Regression predicts a numeric value, while classification predicts a category. 
2. A confusion table counts how many predictions a specific model got right and the mistakes it made. Correct predictions show on the diagonal, while the mistakes show off the diagonal. This table shows which classes the model accuractly predicts and which class it doesn't predict well. 
3. The SSE quantifies the total squared difference between the predicted and actual numeric values. This shows how prediction's accuracy range to actual values on the evaluated data. 
4. Overfitting happens when a model learns more details specific to the training data, resulting in it performing well on training data; however, worse on new data. Underfitting happens when a model doesn't capture enough patterns and performs badly on both. 
5. These things improve model performance because splitting the data allows us to evaluate the model on other data that has not been used for training.
6. For a class label, a potential strength is that it is simple to understand and use, while its weakness is that it doesn't show a confidence level for the model. For a probability distribution, its strength is that it shows uncertainty and potential classes, while its weakness is that it doesn't give a clear decision. So, it relies on someone to decide how to use those probabilities. 

### Question 2.1
Loads `land_mines.csv` dataset and checks the class distribution and feature ranges. 

In [1]:
import pandas as pd

df = pd.read_csv('data/land_mines.csv')
print(df.head())
print('\nShape:', df.shape)
print('\nColumns:', list(df.columns))
print('\nTarget counts:\n', df['mine_type'].value_counts().sort_index())
print('\nFeature summary:\n')
print(df.describe().round(3).to_string())

    voltage    height  soil  mine_type
0  0.338157  0.000000   0.0          1
1  0.320241  0.181818   0.0          1
2  0.287009  0.272727   0.0          1
3  0.256284  0.454545   0.0          1
4  0.262840  0.545455   0.0          1

Shape: (338, 4)

Columns: ['voltage', 'height', 'soil', 'mine_type']

Target counts:
 mine_type
1    71
2    70
3    66
4    66
5    65
Name: count, dtype: int64

Feature summary:

       voltage   height     soil  mine_type
count  338.000  338.000  338.000    338.000
mean     0.431    0.509    0.504      2.953
std      0.196    0.306    0.344      1.420
min      0.198    0.000    0.000      1.000
25%      0.310    0.273    0.200      2.000
50%      0.360    0.545    0.600      3.000
75%      0.483    0.727    0.800      4.000
max      1.000    1.000    1.000      5.000


### Question 2.2
Split the data 50/50 into training and testing sets.

In [5]:
from sklearn.model_selection import train_test_split

x = df[['voltage', 'height', 'soil']]
y = df['mine_type']

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.5,
    random_state=42,
    stratify=y
)

print('Training shape:', x_train.shape)
print('Test shape:', x_test.shape)
print('\nTraining class counts:\n', y_train.value_counts().sort_index())
print('\nTest class counts:\n', y_test.value_counts().sort_index())

Training shape: (169, 3)
Test shape: (169, 3)

Training class counts:
 mine_type
1    35
2    35
3    33
4    33
5    33
Name: count, dtype: int64

Test class counts:
 mine_type
1    36
2    35
3    33
4    33
5    32
Name: count, dtype: int64


### Questions 2.3–2.4
Tests different values of `k`, selects the most accurate model, and evaluates it with a confusion matrix.

In [8]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

results = []
for k in range(1, 26):
    knn_model = KNeighborsClassifier(n_neighbors=k)
    knn_model.fit(x_train, y_train)
    predicted_labels = knn_model.predict(x_test)
    accuracy = accuracy_score(y_test, predicted_labels)
    results.append((k, accuracy))

best_k, best_accuracy = max(results, key=lambda x: x[1])
print('All k values tested:\n')
for k, acc in results:
    print(f'k = {k:2d}  accuracy = {acc:.4f}')

print('\nBest k:', best_k)
print('Best accuracy:', round(best_accuracy, 4))

selected_model = KNeighborsClassifier(n_neighbors=best_k)
selected_model.fit(x_train, y_train)
predicted_labels = selected_model.predict(x_test)

accuracy = accuracy_score(y_test, predicted_labels)
confusion_counts = confusion_matrix(y_test, predicted_labels, labels=[1, 2, 3, 4, 5])
confusion_table = pd.DataFrame(confusion_counts, index=[1, 2, 3, 4, 5], columns=[1, 2, 3, 4, 5])

print('\nOverall test accuracy:', round(accuracy, 4))
print('\nConfusion matrix (rows = actual, columns = predicted):\n')
print(confusion_table)

print('\nPer-class recall:\n')
for cls in [1, 2, 3, 4, 5]:
    idx = cls - 1
    tp = confusion_counts[idx, idx]
    total = confusion_counts[idx].sum()
    recall = tp / total if total else 0
    print(f'Class {cls}: recall = {recall:.3f}')

All k values tested:

k =  1  accuracy = 0.4024
k =  2  accuracy = 0.4379
k =  3  accuracy = 0.3787
k =  4  accuracy = 0.3550
k =  5  accuracy = 0.3136
k =  6  accuracy = 0.3254
k =  7  accuracy = 0.3136
k =  8  accuracy = 0.3077
k =  9  accuracy = 0.3491
k = 10  accuracy = 0.3373
k = 11  accuracy = 0.3550
k = 12  accuracy = 0.3373
k = 13  accuracy = 0.3491
k = 14  accuracy = 0.3136
k = 15  accuracy = 0.3314
k = 16  accuracy = 0.3432
k = 17  accuracy = 0.3609
k = 18  accuracy = 0.3373
k = 19  accuracy = 0.3373
k = 20  accuracy = 0.3432
k = 21  accuracy = 0.3609
k = 22  accuracy = 0.3491
k = 23  accuracy = 0.3254
k = 24  accuracy = 0.3432
k = 25  accuracy = 0.3550

Best k: 2
Best accuracy: 0.4379

Overall test accuracy: 0.4379

Confusion matrix (rows = actual, columns = predicted):

    1   2   3  4  5
1  25   0   6  4  1
2   0  32   0  3  0
3  12   2   9  6  4
4  12   5   8  8  0
5  15   2  10  5  0

Per-class recall:

Class 1: recall = 0.694
Class 2: recall = 0.914
Class 3: recall = 0

### Question 2.5: Use the model in practice

I would not use this model to decide how to handle or remove a mine because it makes too many mistakes. It correctly identifies 32 of 35 type 2 mines, but it also labels 9 other mines as type 2. It identifies only 9 of 33 type 3 mines, 8 of 33 type 4 mines, and none of the type 5 mines. A trained expert and approved safety methods should always guide decisions. The model would need to improve and be tested more before it could be useful in practice.

## *Phase 2: AI-assisted Revision
Keep Phase 1 frozen.

- **Phase 1 commit SHA ID:** [To be provided]
- **AI tool and model used:** [To be provided]

### *Prompts used

1. [Paste prompt]
2. [Paste follow-up prompt]

### *AI-proposed changes


[Paste the AI's concise numbered list verbatim.]

1. ...
2. ...

### *Evaluation of AI suggestions


| Change | Decision | Reason and verification |
|---|---|---|
| 1 | Accept / modify / reject | ... |
| 2 | Accept / modify / reject | ... |

### *AI-assisted solution in full
After the evaluation of AI suggestions, incorporate the accepted revisions into your Phase 1 solution and provide the final Phase 2 solution in full in the following cells.

In [ ]:
# Your Phase 2 solution to the problem goes here.


### *Reflection on AI assistance
In your own words without generative AI.

[In 150–300 words, describe the main improvements, AI mistakes or limitations, how the final solution was verified, and any remaining concerns.]